# CSI v2：有效区域有人／无人
本版0=区域无人，1=区域有人；风扇及门外走动属于0。与v1运动检测任务不同。
门始终打开，非活动人员在区域外。按文件名和用户确认信息赋标签，原始metadata不作为标签依据。
先读下方说明，再依次运行。

# CSI v2：有效检测区域有人／无人

这是已训练的探索版模型，替换v1的任务定义。v1保留，不覆盖。首次训练在助手运行环境执行；没有在用户Google账号部署服务，没有接入官网实时流。CPU即可训练。代码可以在本地或Colab复现。

## 本次变更
- 目标：固定房间6B514内，预先规定的有效检测区域内有人=1，无人=0；范围外人员不算有人。区域边界尚未测量，不能根据模型检测结果动态改边界。
- 文件名与用户后续说明为场景标签依据，优先于文件内position/action/initial_label；保留原始文件，不修改原始标签。
- 空场、风扇无人、门外走动：0。区域内走动、玩电脑、挥手：1。打字停顿不自动变成无人。
- 起身出教室：缺少离开有效区域的时间，排除整份记录，待标注后加入。
- 第一批门始终打开，每次最多一个运动物体；有人走动时风扇关闭；非活动组员处于有效范围之外。不把其他场景风扇状态自动补写为关闭。
- 门口走发生在房间外门附近；照片与位置说明存于先前的场地记录。本模型不以照片或文件名作为输入特征。

## 怎么在本地重跑
推荐使用Python 3.12。将原来的采集数据.zip复制到本文件夹；不要解压或改名其中JSONL文件。终端进入本文件夹后执行：

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
.\.venv\Scripts\python.exe train.py --zip "采集数据.zip" --manifest labels.json --out results_local
```

如果未安装Python 3.12，先配置该版本；不要直接改动全局Python环境。再次运行相同输出目录会覆盖该目录里的结果，保留实验时请换输出目录名。

预测新的完整记录：

```powershell
.\.venv\Scripts\python.exe train.py --predict "新记录.jsonl" --model results/occupancy_model.joblib --predictions predictions.csv
```

预测读取JSONL数值，不靠文件名确定答案。当前支持64子载波、约100Hz的数据；输入需有完整metadata和summary。不能直接打开joblib运行；需与train.py配套。仅加载可信来源的joblib。

## 怎么在云端重跑
打开 https://colab.research.google.com/ → 文件 → 上传笔记本 → 选择 CSI_Occupancy_v2.ipynb。按顺序运行；上传数据时选择原来的采集数据.zip。默认CPU即可。模型与结果要下载保存，Colab临时运行环境会回收。笔记本可分享给组员；每个人独立训练，不会自动合并模型。

## 方法与划分
保持v1算法和参数不变，仅更新任务、标签和参与记录，避免把这次任务纠正混同为算法提升。
随机森林200棵树、最大深度5、叶节点至少3个样本、类权重平衡、随机种子42。由I/Q重建幅度，每段2秒，首尾各裁掉1秒。每子载波相对幅度的标准差、相邻帧绝对差均值、极差、差值标准差，共256维；无全数据拟合预处理。低质量窗口舍弃，不补帧。具体规则见train.py。

8份训练记录：空场1、风扇1/2、走动1/2、玩电脑1/2、挥手1。
6份开发留出记录：空场2、风扇3、走动3、玩电脑3、挥手2、门口走。
1份排除：起身出教室。
同一记录不跨组，重复记录和跨组时间重叠会报错。标签和划分见labels.json。

门口走只有一份，因此本次作为未见场景的开发检查，不参与训练。这不代表模型已经学习或验证了所有门外干扰。

## 实测结果
过滤后训练82个窗口，开发留出68个窗口；每窗口2秒。

| 场景 | 正确窗口/总窗口 | 正确率 |
|---|---:|---:|
| 空场无人2 | 2/12 | 16.7% |
| 风扇无人3 | 7/11 | 63.6% |
| 门外走动 | 10/10 | 100% |
| 玩电脑3 | 12/12 | 100% |
| 挥手2 | 10/10 | 100% |
| 走动3 | 12/13 | 92.3% |

总体正确53/68，即77.9%；平衡准确率77.4%。无人33个窗口中14个误报有人；有人35个窗口中1个漏报。训练集多数类恒定预测基线正确率51.5%。
当前空场误报严重，尚不可靠，不应作为验收通过版本。单场景100%只反映一份短记录。
此批数据已用于v1开发和检查，本次属于开发留出评估，不是全新盲测。v1判断运动，v2判断区域占用，标签及样本不同，两次正确率不能直接比较。没有基于本次结果反复调参或选择最佳种子。

## 已知限制与下一步
数据只有同一设备c5628c、同一天、同一房间的一批短记录。标签按用户说明确认，但无同步视频精确核对边界时刻。2秒窗口和首尾裁切无法保证消除过渡。
区域边界需事先落地标记；当前模型只学习样本关系，没有显式定位或距离测量能力，不能保证区分所有区域内外位置。未验证长时间静坐、睡眠、多人或人与风扇同时运动。
下一轮优先补多次独立空场与风扇无人记录，再补区域内静坐和区域外活动；固定设备布局，记录区域边界，留整次新实验作为最终测试。

## 输出文件
- results/occupancy_model.joblib：已训练v2模型及任务配置。
- results/report.json：真实指标、实验条件、版本、数据与标签哈希。
- results/audit.csv：逐记录质量检查，原始标签仅用于审计。
- results/test_predictions.csv：开发留出窗口的标签和预测。
- labels.json：按文件名指定标签、划分、排除理由。
- train.py：训练与推理共享特征实现。
- CSI_Occupancy_v2.ipynb：自包含Colab训练文件。

输出occupancy_score是随机森林类别分数，未校准为可靠概率。数据包不重复附送原始采集数据或照片。


In [ ]:
%pip -q install numpy==2.3.5 scikit-learn==1.8.0 joblib==1.5.3

## 上传原始采集数据.zip
这会将数据上传到你的Google Colab环境。不要选择模型工具包ZIP。

In [ ]:
from google.colab import files
from pathlib import Path
import json
uploaded=files.upload()
archives=[n for n in uploaded if n.lower().endswith(".zip")]
assert len(archives)==1,"请仅上传一个原始数据ZIP"
archive=archives[0]
del uploaded

## 标签与固定划分
新数据请先审查标签和按实验分组，不能自动将未知文件归为某一类。

In [ ]:
labels=json.loads('[\n  {\n    "file": "玩电脑（1）.jsonl",\n    "label": 1,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "玩电脑（2）.jsonl",\n    "label": 1,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "玩电脑（3）.jsonl",\n    "label": 1,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "空场无人（1）.jsonl",\n    "label": 0,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "空场无人（2）.jsonl",\n    "label": 0,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "设备旁挥手(2).jsonl",\n    "label": 1,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "设备旁挥手（1）.jsonl",\n    "label": 1,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "走动（1）.jsonl",\n    "label": 1,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "走动（2）.jsonl",\n    "label": 1,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "走动（3）.jsonl",\n    "label": 1,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "起身出教室.jsonl",\n    "label": null,\n    "split": "exclude",\n    "reason": "Crosses occupancy states; exact effective-zone exit time unknown"\n  },\n  {\n    "file": "门口走.jsonl",\n    "label": 0,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "风扇无人（1）.jsonl",\n    "label": 0,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "风扇无人（2）.jsonl",\n    "label": 0,\n    "split": "train",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  },\n  {\n    "file": "风扇无人（3).jsonl",\n    "label": 0,\n    "split": "test",\n    "reason": "Filename authoritative per user; in-zone person=1, fan-only/empty/outside-door person=0; no synchronized video"\n  }\n]')
print(json.dumps(labels,ensure_ascii=False,indent=2))
Path("labels.json").write_text(json.dumps(labels,ensure_ascii=False,indent=2),encoding="utf-8")

In [ ]:
Path("train.py").write_text('"""Exploratory effective-zone occupancy baseline. User-confirmed filename labels; no synchronized video."""\nimport argparse, csv, hashlib, json, platform, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport sklearn\nimport joblib\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix\n\nWINDOW = 2.0\nTRIM = 1.0\n\ndef read_record(blob):\n    rows = [json.loads(line) for line in blob.splitlines() if line.strip()]\n    meta = [r for r in rows if r.get(\'type\') == \'metadata\']\n    summaries = [r for r in rows if r.get(\'type\') == \'summary\']\n    frames = [r for r in rows if r.get(\'type\') == \'csi\']\n    if len(meta) != 1 or len(summaries) != 1 or not frames:\n        raise ValueError(\'Missing/duplicate metadata or summary, or empty recording\')\n    summary = summaries[0]\n    if summary[\'status\'] != \'complete\' or summary[\'saved_records\'] != len(frames):\n        raise ValueError(\'Recording not complete or frame count mismatch\')\n    ticks = [int(r[\'device_capture_us\']) for r in frames]\n    t = np.array([(x-ticks[0])/1e6 for x in ticks])\n    seq = np.array([int(r[\'sequence\']) for r in frames], dtype=np.int64)\n    if np.any(np.diff(t) <= 0) or np.any(np.diff(seq) <= 0):\n        raise ValueError(\'Time/sequence is not strictly increasing\')\n    if any(r[\'subcarrier_count\'] != 64 for r in frames):\n        raise ValueError(\'v1 supports 64 subcarriers only\')\n    i = np.array([r[\'i\'] for r in frames], dtype=float)\n    q = np.array([r[\'q\'] for r in frames], dtype=float)\n    if i.shape != (len(frames),64) or q.shape != i.shape:\n        raise ValueError(\'Invalid I/Q shape\')\n    a = np.hypot(i,q)\n    if not np.isfinite(a).all():\n        raise ValueError(\'Nonfinite samples\')\n    audit = dict(frames=len(frames), duration_s=float(t[-1]),\n                 missing_sequences=int(np.sum(np.diff(seq)-1)),\n                 max_gap_s=float(np.max(np.diff(t))), device=meta[0].get(\'device\'),\n                 initial_label=meta[0].get(\'initial_label\'), label_verified=meta[0].get(\'label_verified\'))\n    return meta[0], t, seq, a, audit\n\ndef features(a):\n    # Per-window normalization, not a fitted transform. Never use test data to fit preprocessing.\n    relative = a / np.maximum(a.mean(axis=0), 1.0)\n    delta = np.diff(relative, axis=0)\n    return np.concatenate([relative.std(axis=0), np.mean(np.abs(delta),axis=0),\n                           np.ptp(relative,axis=0), delta.std(axis=0)])\n\ndef windows(t, seq, a):\n    out = []\n    rejected = 0\n    # Non-overlapping windows; trim the edges, without claiming transitions are thereby removed.\n    for start in np.arange(TRIM, t[-1]-TRIM-WINDOW+1e-8, WINDOW):\n        ix = np.flatnonzero((t >= start) & (t < start+WINDOW))\n        if (len(ix) < 180 or len(ix) > 220 or t[ix[-1]]-t[ix[0]] < 1.9\n            or np.max(np.diff(t[ix])) > .05 or np.any(np.diff(seq[ix]) != 1)):\n            rejected += 1\n            continue\n        out.append((float(start),features(a[ix])))\n    return out,rejected\n\ndef write_csv(path, rows):\n    if not rows: return\n    with open(path,\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n\ndef train(archive, manifest, output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    entries=json.loads(Path(manifest).read_text(encoding=\'utf-8\'))\n    samples=[]; audits=[]; ids={}; fingerprints={}; host_intervals=[]\n    with zipfile.ZipFile(archive) as z:\n        files={Path(n).name:n for n in z.namelist() if n.endswith(\'.jsonl\')}\n        if len(files)!=sum(n.endswith(\'.jsonl\') for n in z.namelist()):\n            raise ValueError(\'Duplicate filenames\')\n        for item in entries:\n            blob=z.read(files[item[\'file\']]);meta,t,seq,a,audit=read_record(blob)\n            audit.update(file=item[\'file\'],split=item[\'split\'],label=item[\'label\'],reason=item[\'reason\'])\n            audits.append(audit)\n            if item[\'split\']==\'exclude\':continue\n            if item[\'label\'] not in (0,1) or item[\'split\'] not in (\'train\',\'test\'):\n                raise ValueError(\'Invalid label or split\')\n            for value,seen in [(meta[\'session_id\'],ids),(hashlib.sha256(blob).hexdigest(),fingerprints)]:\n                if value in seen: raise ValueError(\'Duplicate included recording\')\n                seen[value]=item[\'split\']\n            # Require simultaneous views to share a split; host wall clock is used for this check only.\n            from datetime import datetime\n            start=datetime.fromisoformat(meta[\'started_utc\'].replace(\'Z\',\'+00:00\')).timestamp()\n            end=start+t[-1]\n            for s,e,split in host_intervals:\n                if min(end,e)>max(start,s) and item[\'split\']!=split:\n                    raise ValueError(\'Overlapping sessions across train/test: group these together\')\n            host_intervals.append((start,end,item[\'split\']))\n            ws,rejected=windows(t,seq,a)\n            audit.update(accepted_windows=len(ws),rejected_windows=rejected)\n            for start,x in ws:samples.append(dict(file=item[\'file\'],split=item[\'split\'],label=item[\'label\'],start_s=start,x=x))\n    for audit in audits:\n        audit.setdefault(\'accepted_windows\',0);audit.setdefault(\'rejected_windows\',0)\n    write_csv(output/\'audit.csv\',audits)\n    tr=[r for r in samples if r[\'split\']==\'train\'];te=[r for r in samples if r[\'split\']==\'test\']\n    if {r[\'label\'] for r in tr}!={0,1} or {r[\'label\'] for r in te}!={0,1}:\n        raise ValueError(\'Both train and test must contain both classes after quality filtering\')\n    assert not {r[\'file\'] for r in tr}&{r[\'file\'] for r in te}\n    model=RandomForestClassifier(n_estimators=200,max_depth=5,min_samples_leaf=3,\n                                  class_weight=\'balanced\',random_state=42,n_jobs=-1)\n    model.fit(np.array([r[\'x\'] for r in tr]),[r[\'label\'] for r in tr])\n    x_test=np.array([r[\'x\'] for r in te]);y=np.array([r[\'label\'] for r in te])\n    pred=model.predict(x_test);p=model.predict_proba(x_test)[:,list(model.classes_).index(1)]\n    predictions=[dict(file=r[\'file\'],start_s=r[\'start_s\'],end_s=r[\'start_s\']+WINDOW,\n                      assigned_label=r[\'label\'],prediction=int(h),occupancy_score=float(v))\n                 for r,h,v in zip(te,pred,p)]\n    write_csv(output/\'test_predictions.csv\',predictions)\n    report=dict(status=\'development_evaluation_not_final\', task=\'effective_zone_occupancy\', label_source=\'user_confirmed_filenames_and_context\', evaluation_role=\'reused_batch_development_holdout\',train_windows=len(tr),test_windows=len(te),\n                train_files=sorted({r[\'file\'] for r in tr}),test_files=sorted({r[\'file\'] for r in te}),\n                accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),\n                confusion_matrix_rows_true_cols_pred_0_1=confusion_matrix(y,pred,labels=[0,1]).tolist(),\n                majority_train_baseline_accuracy=float(np.mean(y==int(np.bincount([r[\'label\'] for r in tr]).argmax()))),\n                per_file={f:dict(windows=sum(r[\'file\']==f for r in predictions),\n                            accuracy=float(np.mean([r[\'prediction\']==r[\'assigned_label\'] for r in predictions if r[\'file\']==f])))\n                          for f in sorted({r[\'file\'] for r in te})},\n                limitations=[\'Labels assigned from user-confirmed filenames and scene context; exact transitions are not video-verified.\',\n                  \'Only one device, room and collection day; only two empty-room recordings.\',\n                  \'Target is occupancy within the designated effective zone, not the whole room. Fan-only and outside-door movement are negative. No verified long stationary/sleeping-person evaluation.\',\n                  \'This batch was previously inspected and evaluated. Holdout results are development evidence, not a new blind final test. Zone boundary coordinates remain unspecified.\',\n                  \'Occupancy score is not a calibrated probability. Architecture and hyperparameters unchanged from v1; no tuning on this evaluation.\'],\n                versions=dict(python=platform.python_version(),numpy=np.__version__,sklearn=sklearn.__version__,joblib=joblib.__version__),\n                experiment_conditions=dict(door=\'open throughout batch 1\',max_moving_objects=1,fan_during_person_walking=\'off\',nonparticipants=\'outside effective zone\',door_walk=\'outside room near door\',zone_boundary=\'not measured\'),\n                labels_manifest_sha256=hashlib.sha256(Path(manifest).read_bytes()).hexdigest(),\n                archive_sha256=hashlib.sha256(Path(archive).read_bytes()).hexdigest())\n    (output/\'report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    joblib.dump(dict(model=model,window_s=WINDOW,trim_s=TRIM,feature_version=\'relative_amplitude_v1\',\n                     task=\'effective_zone_occupancy\', labels={0:\'unoccupied\',1:\'occupied\'},report=report),output/\'occupancy_model.joblib\',compress=3)\n    print(json.dumps(report,ensure_ascii=False,indent=2))\n    return report\n\ndef predict(recording, model_path, output):\n    # Only load trusted joblib files: pickle-based formats can execute code.\n    bundle=joblib.load(model_path)\n    if bundle.get(\'task\') != \'effective_zone_occupancy\':raise ValueError(\'This predictor requires an occupancy v2 model, not motion v1\')\n    if bundle[\'feature_version\']!=\'relative_amplitude_v1\':raise ValueError(\'Unknown feature version\')\n    _,t,seq,a,_=read_record(Path(recording).read_bytes());ws,rejected=windows(t,seq,a)\n    if not ws:raise ValueError(\'No usable windows (needs complete 64-subcarrier recording near 100 Hz)\')\n    model=bundle[\'model\'];x=np.array([w[1] for w in ws]);pr=model.predict(x)\n    score=model.predict_proba(x)[:,list(model.classes_).index(1)]\n    write_csv(output,[dict(start_s=w[0],end_s=w[0]+WINDOW,prediction=int(y),label=bundle[\'labels\'][int(y)],occupancy_score=float(s)) for w,y,s in zip(ws,pr,score)])\n    print(f\'Saved {len(ws)} predictions; rejected {rejected} windows\')\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--zip\');p.add_argument(\'--manifest\',default=\'labels.json\');p.add_argument(\'--out\',default=\'results\')\n    p.add_argument(\'--predict\');p.add_argument(\'--model\',default=\'results/occupancy_model.joblib\');p.add_argument(\'--predictions\',default=\'predictions.csv\');args=p.parse_args()\n    if args.predict:predict(args.predict,args.model,args.predictions)\n    elif args.zip:train(args.zip,args.manifest,args.out)\n    else:p.error(\'Use --zip for training or --predict for inference\')\n',encoding="utf-8")

## 训练和开发留出评估
本批已参与此前开发。模型保存到results，结果不能作为新盲测成绩。

In [ ]:
import importlib,train
importlib.reload(train)
report=train.train(archive,"labels.json","results")

In [ ]:
import shutil
shutil.make_archive("CSI_v2_results","zip","results")
files.download("CSI_v2_results.zip")

## 可选：预测新JSONL记录
无需设置正确答案即可预测；评估预测对错需另行标注。

In [ ]:
new_files=files.upload()
for n in new_files:
    if n.endswith(".jsonl"):
        out=Path(n).stem+"_occupancy.csv"
        train.predict(n,"results/occupancy_model.joblib",out)
        files.download(out)